# Data Understanding & Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Load dataset
df = pd.read_csv('data/Distribusi_Pupuk_Jatim_2023-2025.csv')

# Convert numeric columns
kolom_numeric = [
    'luas_panen_kedelai',
    'produksi_kedelai_ton',
]

def clean_and_convert(series):
    series = series.astype(str)
    series = series.str.replace(',', '.')
    series = series.str.strip()
    return pd.to_numeric(series, errors='coerce')

df[kolom_numeric] = df[kolom_numeric].apply(clean_and_convert)

print(f"Ukuran Dataset: {df.shape[0]} baris, {df.shape[1]} kolom.")
print("\nKolom: ", df.columns.tolist())
print("\nTipe data per kolom:")
print(df.dtypes)
print("\nInfo Dataset: ")
df.head()

In [ ]:
# Verify data quality
duplicates = df.duplicated(subset=['kabupaten', 'bulan', 'tahun'], keep=False)
print(f"\nJumlah duplikat (kabupaten+bulan+tahun): {duplicates.sum()}")

if duplicates.sum() > 0:
    print("Contoh data duplikat:")
    print(df[duplicates].sort_values(['kabupaten', 'tahun', 'bulan']).head(10))
else:
    print("Tidak ditemukan data duplikat.")

print("\nMissing values per kolom:")
missing_value = df.isnull().sum()
print(missing_value)

print("\nNilai 0 per kolom:")
numeric_cols = df.select_dtypes(include=[np.number]).columns
zero_counts = (df[numeric_cols] == 0).sum()
zero_df = pd.DataFrame({
    'Jumlah Nilai 0': zero_counts,
    'Persentase (%)': (zero_counts / len(df) * 100).round(2)
})
zero_df = zero_df[zero_df['Jumlah Nilai 0'] > 0].sort_values('Jumlah Nilai 0', ascending=False)

if len(zero_df) > 0:
    print(zero_df)
else:
    print("Tidak ada kolom dengan nilai 0")

print("\nKabupaten dengan data tidak lengkap:")
kabupaten_completeness = df.groupby('kabupaten').size()
incomplete_kab = kabupaten_completeness[kabupaten_completeness < 36]
if len(incomplete_kab) > 0:
    print(f"Jumlah kabupaten tidak lengkap: {len(incomplete_kab)}")
    for kab, count in incomplete_kab.items():
        print(f"  - {kab}: {count} dari 36 periode")
else:
    print("Semua kabupaten memiliki 36 periode data")

In [ ]:
# Descriptive statistics
fitur_pupuk = ['alokasi_urea', 'distribusi_urea', 'alokasi_organik',
               'distribusi_organik', 'alokasi_za', 'distribusi_za',
               'alokasi_NPK', 'distribusi_NPK']
fitur_iklim = ['curah_hujan', 'suhu']
fitur_padi = ['luas_panen_padi', 'produksi_padi_ton']
fitur_jagung = ['luas_panen_jagung', 'produksi_jagung_ton']
fitur_kedelai = ['luas_panen_kedelai', 'produksi_kedelai_ton']

semua_numerik = fitur_pupuk + fitur_iklim + fitur_padi + fitur_jagung + fitur_kedelai

print("=== SKEWNESS SELURUH FITUR NUMERIK ===")
skew_df = pd.DataFrame({
    'Skewness': df[semua_numerik].skew().sort_values(ascending=False)
})
print(skew_df)
print("\nInterpretasi:")
print("- |Skewness| < 0.5 : approximately symmetric")
print("- 0.5 <= |Skewness| < 1 : moderately skewed")
print("- |Skewness| >= 1 : highly skewed (pertimbangkan log-transform)")

print("\nStatistik deskriptif:")
df.describe()

In [ ]:
# EDA - Distribution of fertilizer features
fig, axes = plt.subplots(4, 2, figsize=(15, 20))
axes = axes.flatten()

for i, col in enumerate(fitur_pupuk):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], bins=30)
    mean_val = df[col].mean()
    median_val = df[col].median()
    axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f}')
    axes[i].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.2f}')
    axes[i].legend()

plt.tight_layout()
plt.suptitle('Distribusi Alokasi & Distribusi Pupuk', fontsize=16, y=1.02)
plt.show()

# Distribution of climate features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(fitur_iklim):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], bins=30)
    mean_val = df[col].mean()
    median_val = df[col].median()
    axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f}')
    axes[i].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.2f}')
    axes[i].legend()

plt.tight_layout()
plt.suptitle('Distribusi Faktor Iklim', fontsize=14, y=1.05)
plt.show()

In [ ]:
# EDA - Spatial trends analysis
kabupaten_agg = df.groupby('kabupaten').agg({
    'produksi_padi_ton': 'mean',
    'produksi_jagung_ton': 'mean',
    'produksi_kedelai_ton': 'mean',
    'distribusi_urea': 'mean',
    'distribusi_NPK': 'mean',
    'distribusi_organik': 'mean',
    'distribusi_za': 'mean'
}).round(2)

kabupaten_agg = kabupaten_agg.reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 8))

colors_padi = sns.color_palette("Greens_r", 10)
colors_jagung = sns.color_palette("Oranges_r", 10)
colors_kedelai = sns.color_palette("Blues_r", 10)

top10_padi = kabupaten_agg.nlargest(10, 'produksi_padi_ton')[['kabupaten', 'produksi_padi_ton']]
top10_jagung = kabupaten_agg.nlargest(10, 'produksi_jagung_ton')[['kabupaten', 'produksi_jagung_ton']]
top10_kedelai = kabupaten_agg.nlargest(10, 'produksi_kedelai_ton')[['kabupaten', 'produksi_kedelai_ton']]

# Plot 1: Padi
bars1 = axes[0].barh(top10_padi['kabupaten'][::-1], top10_padi['produksi_padi_ton'][::-1],
                     color=colors_padi, edgecolor='darkgreen', linewidth=0.8)
axes[0].set_title('TOP 10 PRODUKSI PADI', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Rata-rata Produksi (Ton)', fontsize=10)

for bar, val in zip(bars1, top10_padi['produksi_padi_ton'][::-1]):
    axes[0].text(bar.get_width() + max(top10_padi['produksi_padi_ton'])*0.01,
                 bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f}', va='center', fontsize=9, fontweight='bold')

# Plot 2: Jagung
bars2 = axes[1].barh(top10_jagung['kabupaten'][::-1], top10_jagung['produksi_jagung_ton'][::-1],
                     color=colors_jagung, edgecolor='darkorange', linewidth=0.8)
axes[1].set_title('TOP 10 PRODUKSI JAGUNG', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Rata-rata Produksi (Ton)', fontsize=10)

for bar, val in zip(bars2, top10_jagung['produksi_jagung_ton'][::-1]):
    axes[1].text(bar.get_width() + max(top10_jagung['produksi_jagung_ton'])*0.01,
                 bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f}', va='center', fontsize=9, fontweight='bold')

# Plot 3: Kedelai
bars3 = axes[2].barh(top10_kedelai['kabupaten'][::-1], top10_kedelai['produksi_kedelai_ton'][::-1],
                     color=colors_kedelai, edgecolor='darkblue', linewidth=0.8)
axes[2].set_title('TOP 10 PRODUKSI KEDELAI', fontsize=14, fontweight='bold', pad=15)
axes[2].set_xlabel('Rata-rata Produksi (Ton)', fontsize=10)

for bar, val in zip(bars3, top10_kedelai['produksi_kedelai_ton'][::-1]):
    axes[2].text(bar.get_width() + max(top10_kedelai['produksi_kedelai_ton'])*0.01,
                 bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f}', va='center', fontsize=9, fontweight='bold')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.suptitle('LUMBUNG PANGAN JAWA TIMUR: TOP 10 KABUPATEN PER KOMODITAS',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(18, 16))

top10_urea = kabupaten_agg.nlargest(10, 'distribusi_urea')[['kabupaten', 'distribusi_urea']]
top10_NPK = kabupaten_agg.nlargest(10, 'distribusi_NPK')[['kabupaten', 'distribusi_NPK']]
top10_organik = kabupaten_agg.nlargest(10, 'distribusi_organik')[['kabupaten', 'distribusi_organik']]
top10_za = kabupaten_agg.nlargest(10, 'distribusi_za')[['kabupaten', 'distribusi_za']]

color_urea = ['#1a5276', '#2471a3', '#2980b9', '#3498db', '#5dade2', '#85c1e9', '#aed6f1', '#d4e6f1', '#ebf5fb', '#f4f9f9']
color_NPK = ['#78281f', '#922b21', '#b03a2e', '#c0392b', '#e74c3c', '#f1948a', '#f5b7b1', '#fadbd8', '#fdedec', '#fdf2e9']
color_organik = ['#145a32', '#1e8449', '#239b56', '#27ae60', '#2ecc71', '#58d68d', '#82e0aa', '#abebc6', '#d5f5e3', '#eafaf1']
color_za = ['#4a235a', '#6c3483', '#7d3c98', '#8e44ad', '#a569bd', '#bb8fce', '#d2b4de', '#e8daef', '#f4ecf7', '#f9f4fc']

# Plot 1: Distribusi urea
bars1 = axes[0, 0].barh(top10_urea['kabupaten'][::-1], top10_urea['distribusi_urea'][::-1],
                     color=color_urea, edgecolor='#154360', linewidth=0.8)
axes[0, 0].set_title('DISTRIBUSI UREA', fontsize=14, fontweight='bold', pad=15)
axes[0, 0].set_xlabel('Rata-rata Distribusi (Ton)', fontsize=10)
for bar, val in zip(bars1, top10_urea['distribusi_urea'][::-1]):
    axes[0, 0].text(bar.get_width() + max(top10_urea['distribusi_urea'])*0.01,
                 bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f}', va='center', fontsize=9, fontweight='bold')

# Plot 2: Distribusi NPK
bars2 = axes[0, 1].barh(top10_NPK['kabupaten'][::-1], top10_NPK['distribusi_NPK'][::-1],
                     color=color_NPK, edgecolor='#641e16', linewidth=0.8)
axes[0, 1].set_title('DISTRIBUSI NPK', fontsize=14, fontweight='bold', pad=15)
axes[0, 1].set_xlabel('Rata-rata Distribusi (Ton)', fontsize=10)
for bar, val in zip(bars2, top10_NPK['distribusi_NPK'][::-1]):
    axes[0, 1].text(bar.get_width() + max(top10_NPK['distribusi_NPK'])*0.01,
                 bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f}', va='center', fontsize=9, fontweight='bold')

# Plot 3: Distribusi organik
bars3 = axes[1, 0].barh(top10_organik['kabupaten'][::-1], top10_organik['distribusi_organik'][::-1],
                     color=color_organik, edgecolor='#0b3d0b', linewidth=0.8)
axes[1, 0].set_title('DISTRIBUSI PUPUK ORGANIK', fontsize=14, fontweight='bold', pad=15)
axes[1, 0].set_xlabel('Rata-rata Distribusi (Ton)', fontsize=10)
for bar, val in zip(bars3, top10_organik['distribusi_organik'][::-1]):
    axes[1, 0].text(bar.get_width() + max(top10_organik['distribusi_organik'])*0.01,
                 bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f}', va='center', fontsize=9, fontweight='bold')

# Plot 4: Distribusi za
bars4 = axes[1, 1].barh(top10_za['kabupaten'][::-1], top10_za['distribusi_za'][::-1],
                     color=color_za, edgecolor='#2e0a3d', linewidth=0.8)
axes[1, 1].set_title('DISTRIBUSI ZA', fontsize=14, fontweight='bold', pad=15)
axes[1, 1].set_xlabel('Rata-rata Distribusi (Ton)', fontsize=10)
for bar, val in zip(bars4, top10_za['distribusi_za'][::-1]):
    axes[1, 1].text(bar.get_width() + max(top10_za['distribusi_za'])*0.01,
                 bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f}', va='center', fontsize=9, fontweight='bold')

for ax in axes.flatten():
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.suptitle('ALOKASI PUPUK TERBESAR: TOP 10 KABUPATEN PER JENIS PUPUK',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# EDA - Realization ratio analysis
jenis_pupuk = ["urea", "organik", "za", "NPK"]

for jp in jenis_pupuk:
    alokasi_col = f"alokasi_{jp}"
    distribusi_col = f"distribusi_{jp}"
    rasio_col = f"rasio_realisasi_{jp}"
    df[rasio_col] = np.where(
        df[alokasi_col] > 0,
        df[distribusi_col] / df[alokasi_col],
        np.nan,
    )

rasio_cols = [f"rasio_realisasi_{jp}" for jp in jenis_pupuk]

print("\n=== Statistik Rasio Realisasi per Jenis Pupuk (setelah perbaikan) ===")
print(df[rasio_cols].describe())

for col in rasio_cols:
    n_anomali = (df[col] > 1).sum()
    n_valid = df[col].notna().sum()
    print(f"{col}: rasio > 1 = {n_anomali} dari {n_valid} baris valid ({n_anomali/n_valid*100:.1f}%))

fig.ax = plt.subplots(figsize=(10, 6))
df_melt = df.melt(value_vars=rasio_cols, var_name="jenis_pupuk", value_name="rasio")
df_melt["jenis_pupuk"] = df_melt["jenis_pupuk"].str.replace("rasio_realisasi_", "")

sns.boxplot(data=df_melt[df_melt["rasio"] <= 3], x="jenis_pupuk", y="rasio", ax=ax)
ax.axhline(1.0, color="red", linestyle="--", linewidth=1, label="Rasio = 1 (realisasi penuh)")
ax.set_title("Distribusi Rasio Realisasi Pupuk per Jenis (Distribusi Bulanan/Alokasi)")
ax.set_xlabel("Jenis Pupuk")
ax.set_ylabel("Rasio Realisasi")
ax.legend()
plt.tight_layout()
plt.savefig("01_boxplot_rasio_realisasi.png", dpi=120)
plt.show()

In [ ]:
# EDA - Correlation analysis
korelasi_cols = [
    'curah_hujan', 'suhu',
    'alokasi_urea', 'distribusi_urea',
    'alokasi_NPK', 'distribusi_NPK',
    'alokasi_organik', 'distribusi_organik',
    'alokasi_za', 'distribusi_za',
    'luas_panen_padi', 'produksi_padi_ton',
    'luas_panen_jagung', 'produksi_jagung_ton',
    'luas_panen_kedelai', 'produksi_kedelai_ton'
]

corr_matrix = df[korelasi_cols].corr()

plt.figure(figsize=(20, 16))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Pupuk, Iklim & Produksi', fontsize=18, pad=20)
plt.tight_layout()
plt.show()

# Climate correlation with realization ratio
df["curah_hujan_lag1"] = df.groupby("kabupaten")["curah_hujan"].shift(1)
df["suhu_lag1"] = df.groupby("kabupaten")["suhu"].shift(1)

fitur_korelasi = ["curah_hujan", "suhu", "curah_hujan_lag1", "suhu_lag1"] + rasio_cols
corr_matrix = df[fitur_korelasi].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Korelasi Curah Hujan & Suhu (Saat Ini vs Lag-1 Bulan)\nterhadap Rasio Realisasi Pupuk")
plt.tight_layout()
plt.show()

print("\n=== Korelasi iklim (current & lag-1) vs rasio realisasi ===")
print(corr_matrix.loc[["curah_hujan", "suhu", "curah_hujan_lag1", "suhu_lag1"], rasio_cols])

# Total planted area & productivity vs monthly fertilizer distribution
df["luas_panen_total"] = df["luas_panen_padi"] + df["luas_panen_jagung"] + df["luas_panen_kedelai"]
df["distribusi_total_bulanan"] = (
    df["distribusi_urea"] + df["distribusi_organik"]
    + df["distribusi_za"] + df["distribusi_NPK"]
)

plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df,
    x="luas_panen_total",
    y="distribusi_total_bulanan",
    hue="kabupaten",
    legend=False,
    alpha=0.5
)
plt.title("Luas Panen Total vs Distribusi Pupuk Total (Bulanan)")
plt.xlabel("Luas Panen Total (ha)")
plt.ylabel("Distribusi Pupuk Total Bulanan")
plt.tight_layout()
plt.show()